## SC4001 Group Assignment - Gender Classification
-------
### Done by:
- Mak Chee Cheng
- Ruvenn Siow
- Teo Hong Guan Brian
-------

### 1. Comparing Baseline Levi & Hassner Model with CelebA dataset pretrained model

#### Levi & Hassner CNN 

In [1]:
# Import required libraries
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import cv2
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
import pandas as pd

In [2]:
# Load CelebA dataset
DATASET_PATH = "./img_celeba"
IMAGE_SIZE = (100, 100)
BATCH_SIZE = 32

# Function to create a TensorFlow dataset
def parse_image(filename, label):
    img = tf.io.read_file(filename)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE) / 255.0
    return img, label

# Load file paths and labels
list_attr_path = os.path.join(DATASET_PATH, "list_attr_celeba.txt")
img_paths = []
labels = []

with open(list_attr_path, "r") as f:
    lines = f.readlines()[2:]  # Skip header
    for line in lines:
        parts = line.split()
        img_path = os.path.join(DATASET_PATH, parts[0])
        label = 1 if int(parts[21]) == 1 else 0  # Gender label
        img_paths.append(img_path)
        labels.append(label)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(img_paths, labels, test_size=0.2, random_state=42)

# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

train_dataset = train_dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)

train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [3]:
# Define Levi & Hassner CNN Model
def levi_hassner_cnn(input_shape=(100, 100, 3), num_classes=2):
    model = models.Sequential()
    
    # Conv Layer 1
    model.add(layers.Conv2D(96, (7, 7), strides=4, activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D(pool_size=(3, 3), strides=2))
    model.add(layers.BatchNormalization())
    
    # Conv Layer 2
    model.add(layers.Conv2D(256, (5, 5), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D(pool_size=(3, 3), strides=2))
    model.add(layers.BatchNormalization())
    
    # Conv Layer 3
    model.add(layers.Conv2D(384, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D(pool_size=(3, 3), strides=2))
    model.add(layers.BatchNormalization())
    
    # Fully Connected Layers
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.5))
    
    # Output Layer
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model

In [4]:
# Compile Model
model = levi_hassner_cnn()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train Model with progress tracking
model.fit(train_dataset, epochs=10, validation_data=test_dataset, verbose=1)

# Model Summary
model.summary()

c:\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 766s 150ms/step - accuracy: 0.7587 - loss: 0.5473 - val_accuracy: 0.8607 - val_loss: 0.3150
Epoch 2/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 378s 75ms/step - accuracy: 0.8708 - loss: 0.2996 - val_accuracy: 0.8749 - val_loss: 0.2857
Epoch 3/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 440s 87ms/step - accuracy: 0.8953 - loss: 0.2523 - val_accuracy: 0.8908 - val_loss: 0.2656
Epoch 4/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 401s 79ms/step - accuracy: 0.9107 - loss: 0.2180 - val_accuracy: 0.8793 - val_loss: 0.3132
Epoch 5/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 382s 75ms/step - accuracy: 0.9200 - loss: 0.1900 - val_accuracy: 0.8978 - val_loss: 0.2616
Epoch 6/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 411s 81ms/step - accuracy: 0.9329 - loss: 0.1631 - val_accuracy: 0.8998 - val_loss: 0.3137
Epoch 7/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 390s 77ms/step - accuracy: 0.9429 - loss: 0.1412 - val_accuracy: 0.9023 - val_loss: 0.2629
Epoch 8/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 406s 80ms/step - accuracy:

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 96)     │        14,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 11, 11, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 11, 11, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 5, 5, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 5, 5, 384)      │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 2, 2, 384)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 2, 2, 384)      │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,699,720 (29.37 MB)

 Trainable params: 2,566,082 (9.79 MB)

 Non-trainable params: 1,472 (5.75 KB)

 Optimizer params: 5,132,166 (19.58 MB)

In [6]:
# Save the celebA pretrained model
model.save("celebA_pretrained_model.keras")

In [8]:
from tensorflow.keras.models import load_model

pretrained_model = load_model("celebA_pretrained_model.keras")

In [9]:
# Define paths
DATASET_PATH = "./adience_dataset"  
IMAGE_SIZE = (100, 100)
BATCH_SIZE = 32

def load_adience_dataset(base_path, target_size=(128, 128)):
    """Loads the Adience dataset and preprocesses images."""
    dataframes = []
    
    # Load all fold data files
    for fold in range(5):
        file_path = os.path.join(base_path, f'fold_{fold}_data.txt')
        if os.path.exists(file_path):
            df = pd.read_csv(file_path, sep='\t')
            dataframes.append(df)
    
    # Combine all fold data
    data = pd.concat(dataframes, ignore_index=True)
    
    images, labels = [], []
    for _, row in data.iterrows():
        folder = row['user_id']
        img_name = row['original_image']
        gender = row['gender']
        
        if gender not in ['m', 'f']:  # Skip unknown gender labels
            continue
        
        img_path = os.path.join(base_path, folder, img_name)
        if os.path.exists(img_path):
            img = load_img(img_path, target_size=target_size)
            img_array = img_to_array(img) / 255.0
            
            images.append(img_array)
            labels.append(0 if gender == 'm' else 1)  # 0: Male, 1: Female
    
    # Convert to TensorFlow tensors
    images = tf.convert_to_tensor(images)
    labels = tf.convert_to_tensor(labels)
    
    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)
    
    # Create TensorFlow datasets
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    return train_dataset, test_dataset

In [10]:
# Compile the pretrained model before training on Adience
pretrained_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train on Adience dataset
pretrained_model.fit(train_dataset, epochs=10, validation_data=test_dataset, verbose=1)


Epoch 1/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 382s 75ms/step - accuracy: 0.9711 - loss: 0.0782 - val_accuracy: 0.8670 - val_loss: 0.5400
Epoch 2/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 381s 75ms/step - accuracy: 0.9755 - loss: 0.0677 - val_accuracy: 0.8928 - val_loss: 0.3752
Epoch 3/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 392s 77ms/step - accuracy: 0.9770 - loss: 0.0617 - val_accuracy: 0.8976 - val_loss: 0.4115
Epoch 4/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 399s 79ms/step - accuracy: 0.9792 - loss: 0.0581 - val_accuracy: 0.8973 - val_loss: 0.3630
Epoch 5/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 393s 78ms/step - accuracy: 0.9810 - loss: 0.0534 - val_accuracy: 0.8986 - val_loss: 0.3666
Epoch 6/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 419s 83ms/step - accuracy: 0.9831 - loss: 0.0474 - val_accuracy: 0.8990 - val_loss: 0.3522
Epoch 7/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 374s 74ms/step - accuracy: 0.9845 - loss: 0.0446 - val_accuracy: 0.9005 - val_loss: 0.3812
Epoch 8/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 381s 75ms/step - accuracy: 

In [11]:
base_model = levi_hassner_cnn() # Create a new instance of the model
base_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train from scratch on Adience dataset
base_model.fit(train_dataset, epochs=10, validation_data=test_dataset, verbose=1)

Epoch 1/10


c:\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5065/5065 ━━━━━━━━━━━━━━━━━━━━ 378s 74ms/step - accuracy: 0.7564 - loss: 0.5563 - val_accuracy: 0.8015 - val_loss: 0.4199
Epoch 2/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 376s 74ms/step - accuracy: 0.8691 - loss: 0.3043 - val_accuracy: 0.8633 - val_loss: 0.3140
Epoch 3/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 378s 75ms/step - accuracy: 0.8936 - loss: 0.2530 - val_accuracy: 0.8858 - val_loss: 0.2609
Epoch 4/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 380s 75ms/step - accuracy: 0.9090 - loss: 0.2158 - val_accuracy: 0.8874 - val_loss: 0.2602
Epoch 5/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 381s 75ms/step - accuracy: 0.9232 - loss: 0.1870 - val_accuracy: 0.8962 - val_loss: 0.2550
Epoch 6/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 413s 82ms/step - accuracy: 0.9351 - loss: 0.1572 - val_accuracy: 0.9042 - val_loss: 0.2587
Epoch 7/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 451s 89ms/step - accuracy: 0.9466 - loss: 0.1337 - val_accuracy: 0.8979 - val_loss: 0.2712
Epoch 8/10
5065/5065 ━━━━━━━━━━━━━━━━━━━━ 434s 86ms/step - accuracy: 0.9541 - lo

In [12]:
# Save the baseline model
base_model.save("baseline_model.keras")

In [15]:
pretrained_acc = pretrained_model.evaluate(test_dataset)[1]  # Get accuracy
base_acc = base_model.evaluate(test_dataset)[1]  # Get accuracy

print(f"Pretrained Model Accuracy: {pretrained_acc:.4f}")
print(f"Base Model Accuracy: {base_acc:.4f}")

1267/1267 ━━━━━━━━━━━━━━━━━━━━ 39s 31ms/step - accuracy: 0.8965 - loss: 0.4998
1267/1267 ━━━━━━━━━━━━━━━━━━━━ 38s 30ms/step - accuracy: 0.8966 - loss: 0.2916
Pretrained Model Accuracy: 0.8967
Base Model Accuracy: 0.8931
